# DAEN 328 Assignment 5: Load and Serve

**Name:** _your name here_

Save this notebook as `A5_LastName_FirstName.ipynb`. It must run **top to bottom** (Kernel > Restart & Run All) with all outputs showing when you submit.

Work through `Assignment05_Guide.ipynb` first. Every technique you need is taught there on practice data. This notebook is where you apply it to `EmployeeDB`.

Your dashboard (Part C) is a separate file, `app.py`. See the instructions PDF.

## CONFIG: edit this cell once

In [2]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
import sqlite3
import time
import os 
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_NAME     = os.getenv("DB_NAME")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")

# Part A: Load your cleaned CSV into the 7 EmployeeDB tables

### A1. Read and inspect

Read the CSV with every column as text. Show its shape and the number of blanks per column.

In [3]:
# TODO A1
raw = pd.read_csv("Cleaned_data.csv", dtype=str)
print("The shape of the cleaned data: ",raw.shape, "\n")
df = raw.copy()


The shape of the cleaned data:  (14321, 24) 



In [12]:
raw.isna().sum()

EmployeeID               0
Name                     0
HighestDegree            0
Division                 0
DivisionAddress          0
DivisionPhone            0
DivStreetAddress         0
DivCity                  0
DivState                 0
DivZip                   0
PersonalAddress          0
PerStreetAddress         0
PerCity                  0
PerState                 0
PerZip                   0
Email                    0
PhoneNumber              2
Department               0
DepartmentDescription    0
JobTitle                 1
JobDescription           1
Salary                   1
Birthdate                0
SocialSecurity           0
dtype: int64

### A2. Clean and validate in pandas

Required:
- Strip spaces from text columns.
- Convert `Birthdate` to a real date (`pd.to_datetime(..., errors="coerce").dt.date`) and **print how many could not be read**.
- Convert `Salary` and `EmployeeID` to numbers.
- Put every row that cannot satisfy your schema's NOT NULL or CHECK constraints into a `rejects` DataFrame, save it as `rejects_employees.csv`, and print rows read / good / rejected. Never invent a value (such as a salary of 0) to get a row past a constraint.

In [13]:
# TODO A2

# make copy to clean
df = raw.copy()

# strip spaces
for col in df.columns:
    df[col] = df[col].str.strip()

# convert birthdate to datetime
df["Birthdate"] = pd.to_datetime(df['Birthdate'], format="%m/%d/%Y", errors="coerce").dt.date

# print number of unreadable dates
print(f"Number of birthdates that are unreadbale: {df["Birthdate"].isna().sum()}")

# convert salary and employeeid to numbers
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["EmployeeID"] = pd.to_numeric(df["EmployeeID"], errors="coerce")

Number of birthdates that are unreadbale: 0


In [14]:
# extra step needed: some of the div adresses had commas

for col in ["DivStreetAddress", "DivCity", "PerStreetAddress", "PerCity"]:
    df[col] = df[col].str.rstrip(",").str.strip()

In [15]:
# name (NOT NULL)
bad = df["Name"].isna()

# departmentname (NOT NULL)
bad |= df["Department"].isna()

# jobtitle (NOT NULL)
bad |= df["JobTitle"].isna()

# divtype (NOT NULL)
bad |= df["Division"].isna()

# ssn (NOT NULL)
bad |= df["SocialSecurity"].isna()

# salary (NOT NULL and > 0)
bad |= df["Salary"].isna() | (df["Salary"] <= 0)

# rejects df
rejects = df[bad].copy()
good = df[~bad].copy()

# save and report
rejects.to_csv("rejects_employees.csv", index=False)
print(f"Rows read: {len(df)}   good: {len(good)}   rejected: {len(rejects)}")
rejects[["EmployeeID", "Name",  "JobTitle",  "Salary"]]

Rows read: 14321   good: 14320   rejected: 1


,EmployeeID,Name,JobTitle,Salary
9043,221765,Brittany Monica Sanders,NaN,NaN


### Given: `clean()` (run it, do not change it)

Pass **every** value through `clean()` when you insert it. pandas marks a blank as `NaN`, and if a `NaN` reaches PostgreSQL it is stored as the value `NaN`, not as NULL. A `NaN` salary passes `NOT NULL` and `CHECK (salary > 0)` with no error, and then every average that includes it comes out as `NaN`. `clean()` turns blanks into `None` (a real NULL), so the database rejects what it should. It also turns numpy numbers into plain Python numbers, which avoids the error `can't adapt type numpy.int64`.

In [16]:
def clean(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

# check
print(clean(float("nan")))

None


### Given: start over with empty tables

While you build the loader, a bad attempt can leave wrong rows behind. To empty all 7 tables and restart the id numbers from 1, set `RESET = True`, run the cell, then set it back to `False`. This keeps your tables and constraints; it only removes the rows. Do not use `DROP TABLE`, which deletes your Assignment 4 schema.

Leave `RESET = False` when you submit.

In [ ]:
RESET = False

if RESET:
    with engine.begin() as conn:
        conn.execute(text("""
            TRUNCATE employeedetails, personaldetails, jobdescriptions, divisionlocations,
                     departments, jobtitles, divisions
            RESTART IDENTITY CASCADE;
        """))
    print("All 7 tables emptied.")

All 7 tables emptied.


### A3. The loader

Write `load_employees(good)`. Requirements:
- Loads all 7 tables, lookup tables first, `employeedetails` last, using the CSV-to-column map in the instructions PDF.
- Runs in **one transaction** (all or nothing).
- **Safe to re-run**: a second run adds zero rows to every table.
- Uses `%s` placeholders for every value, and passes every value through `clean()`.

In [38]:
# TODO A3
# this was generated using Claude

def load_employees(good):
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn:                                   # one transaction: all or nothing
            with conn.cursor() as cur:
                # ---- Pattern 1: lookup tables with a UNIQUE name ----
                depts = good[["Department", "DepartmentDescription"]].drop_duplicates("Department")
                for _, r in depts.iterrows():
                    cur.execute(
                        "INSERT INTO departments (departmentname, departmentdescription) VALUES (%s, %s) "
                        "ON CONFLICT (departmentname) DO NOTHING;",
                        (clean(r["Department"]), clean(r["DepartmentDescription"])))

                for name in good["JobTitle"].unique():
                    cur.execute(
                        "INSERT INTO jobtitles (jobtitle) VALUES (%s) "
                        "ON CONFLICT (jobtitle) DO NOTHING;",
                        (clean(name),))
                for name in good["Division"].unique():
                    cur.execute(
                        "INSERT INTO divisions (divtype) VALUES (%s) "
                        "ON CONFLICT (divtype) DO NOTHING;",
                        (clean(name),))

                # ---- Pattern 1: read the generated ids back ----
                cur.execute("SELECT departmentname, departmentid FROM departments;")
                dept_ids = dict(cur.fetchall())
                cur.execute("SELECT jobtitle, jobtitleid FROM jobtitles;")
                jt_ids = dict(cur.fetchall())
                cur.execute("SELECT divtype, divisionid FROM divisions;")
                div_ids = dict(cur.fetchall())

                # ---- Pattern 2: personaldetails, key (SSN) comes from the CSV ----
                people = good.drop_duplicates("SocialSecurity")
                for _, r in people.iterrows():
                    cur.execute(
                        """INSERT INTO personaldetails
                               (ssn, name, highestdegree, email, phonenumber, birthdate,
                                perstreet, percity, perstate, perzip)
                           VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                           ON CONFLICT (ssn) DO NOTHING;""",
                        (clean(r["SocialSecurity"]), clean(r["Name"]), clean(r["HighestDegree"]),
                         clean(r["Email"]), clean(r["PhoneNumber"]), clean(r["Birthdate"]),
                         clean(r["PerStreetAddress"]), clean(r["PerCity"]),
                         clean(r["PerState"]), clean(r["PerZip"])))
                    
                # ---- Pattern 3: divisionlocations, no UNIQUE -> NOT EXISTS on division + street ----
                locs = good[["Division", "DivStreetAddress", "DivCity", "DivState",
                             "DivZip", "DivisionPhone"]] \
                           .drop_duplicates(["Division", "DivStreetAddress"])
                for _, r in locs.iterrows():
                    did = div_ids[r["Division"]]
                    street = clean(r["DivStreetAddress"])
                    cur.execute(
                        """INSERT INTO divisionlocations
                               (divisionid, divstreet, divcity, divstate, divzip, divisionphome)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM divisionlocations
                                             WHERE divisionid = %s AND divstreet = %s);""",
                        (did, street, clean(r["DivCity"]), clean(r["DivState"]),
                         clean(r["DivZip"]), clean(r["DivisionPhone"]), did, street))

                # ---- Pattern 3: jobdescriptions, keyed by TWO ids ----
                jds = good[["JobTitle", "Division", "JobDescription"]] \
                          .drop_duplicates(["JobTitle", "Division"])
                for _, r in jds.iterrows():
                    jid, did = jt_ids[r["JobTitle"]], div_ids[r["Division"]]
                    cur.execute(
                        """INSERT INTO jobdescriptions (jobtitleid, divisionid, description)
                           SELECT %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM jobdescriptions
                                             WHERE jobtitleid = %s AND divisionid = %s);""",
                        (jid, did, clean(r["JobDescription"]), jid, did))
                    
                # ---- Pattern 4: employeedetails, NOT EXISTS on employeeid ----
                for _, r in good.iterrows():
                    emp = clean(r["EmployeeID"])
                    cur.execute(
                        """INSERT INTO employeedetails
                               (employeeid, ssn, departmentid, jobtitleid, divisionid, salary)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM employeedetails WHERE employeeid = %s);""",
                        (emp, clean(r["SocialSecurity"]), dept_ids[r["Department"]],
                         jt_ids[r["JobTitle"]], div_ids[r["Division"]], clean(r["Salary"]), emp))

        print("Load finished and committed.")
    finally:
        conn.close()

In [39]:
# timing for Section D

import time
start = time.perf_counter()
load_employees(good)
end = time.perf_counter()

t = end - start
t

Load finished and committed.


16.893209100002423

### A4. Run it twice and reconcile

Run your loader, record the 7 table counts, run it **again**, record them again. Show one table with, for every table: the count you expected from the CSV, the count after run 1, the count after run 2, and whether all three agree.

Then run the given NaN check below it. Every count must be 0.

In [40]:
# TODO A4

EMP_TABLES = ["departments", "jobtitles", "divisions", "personaldetails",
              "divisionlocations", "jobdescriptions", "employeedetails"]

def table_counts(schema, tables):
    with engine.connect() as conn:
        return {t: conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{t};")).scalar() for t in tables}

load_employees(good)
after_run_1 = table_counts("public", EMP_TABLES)

load_employees(good)                      # second run: should add NOTHING
after_run_2 = table_counts("public", EMP_TABLES)

expected = {
    "departments":       good["Department"].nunique(),
    "jobtitles":         good["JobTitle"].nunique(),
    "divisions":         good["Division"].nunique(),
    "personaldetails":   good["SocialSecurity"].nunique(),
    "divisionlocations": len(good[["Division", "DivStreetAddress"]].drop_duplicates()),
    "jobdescriptions":   len(good[["JobTitle", "Division"]].drop_duplicates()),
    "employeedetails":   len(good),
}

recon = pd.DataFrame({"expected_from_csv": expected,
                      "after_run_1": after_run_1,
                      "after_run_2": after_run_2})
recon["ok"] = (recon["expected_from_csv"] == recon["after_run_1"]) & \
              (recon["after_run_1"] == recon["after_run_2"])

# creating db file
sqlite_conn = sqlite3.connect("EmployeeDB.db")

for t in EMP_TABLES:
    table = pd.read_sql(text(f"SELECT * FROM public.{t};"), engine)
    table.to_sql(t, sqlite_conn, if_exists="replace", index=False)
    print(f"{t}: {len(table)} rows copied")

sqlite_conn.close()

# check
recon

Load finished and committed.
Load finished and committed.
departments: 6 rows copied
jobtitles: 51 rows copied
divisions: 5 rows copied
personaldetails: 14318 rows copied
divisionlocations: 5 rows copied
jobdescriptions: 235 rows copied
employeedetails: 14320 rows copied


,expected_from_csv,after_run_1,after_run_2,ok
departments,6,6,6,True
jobtitles,51,51,51,True
divisions,5,5,5,True
personaldetails,14318,14318,14318,True
divisionlocations,5,5,5,True
jobdescriptions,235,235,235,True
employeedetails,14320,14320,14320,True


In [21]:
# Given: NaN check. Every count must be 0. If not, a value reached the database without clean().
with engine.connect() as conn:
    nan_check = {
        "salary = NaN":    conn.execute(text("SELECT COUNT(*) FROM employeedetails WHERE salary = 'NaN';")).scalar(),
        "name = 'NaN'":    conn.execute(text("SELECT COUNT(*) FROM personaldetails WHERE name = 'NaN';")).scalar(),
        "jobtitle = 'NaN'": conn.execute(text("SELECT COUNT(*) FROM jobtitles WHERE jobtitle = 'NaN';")).scalar(),
    }
nan_check

{'salary = NaN': 0, "name = 'NaN'": 0, "jobtitle = 'NaN'": 0}

### A5. Load report (write in this cell, 4 to 6 sentences)

- How many rows you read, loaded and rejected, and the reason for each kind of reject.
- Which loading pattern you used for `divisionlocations`, `jobdescriptions` and `employeedetails`, and why.
- One real problem you hit while building the loader (the error message or wrong count) and how you found and fixed it.  

##### Answers: 
>- I read in 14321 rows and rejected 1, loading the rest (14320). The rejected row had missing values for both JobTitle and Salary, and JobTitle could not be null due to the "Not null" constraint. 
>- `divisionlocations` and `jobdescriptions` used Pattern 3 because there were no `UNIQUE` constraints and they were child tables. On the other hand, `employeedetails` was a fact table and was uploaded last, therefore it used pattern 4.
>- I had not cleaned the locations previously, and there were duplicates (10 locations rather than the expected 5) of certain cities becasue of extra commas. In addition, while doing the SQL section, I realized that I had forgot to add the divcity and divstate to the divisionlocation tables. After doing that, I had to clean up some of the city names because there were trailing commas creating duplicates for each division location.

# Part B: SQL

Write your own SQL for each question in the code cell under it, using the `q()` helper, and leave the result showing. Rules for every question:
- Filter by **names** (`'Engineering'`, `'Austin'`), never by an id number you looked up.
- Never type a number that the database can compute (an average, a count).
- Show names, not ids, wherever a person would read the result.

In [22]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

**Q1.** Headcount by department: each department's name and number of employees, largest first.

In [23]:
# Q1
q('''
SELECT d.departmentname, COUNT(*) AS employees
FROM employeedetails e
JOIN departments d ON e.departmentid = d.departmentid
GROUP BY d.departmentname
ORDER BY employees DESC;
''')

,departmentname,employees
0,Operations,10580
1,Administrative,2000
2,Engineering,1085
3,Human Resources,645
4,Management,5
5,Executive,5


**Q2.** Engineering employees earning less than $100,000: name, job title and salary, highest salary first.

In [24]:
# Q2
q('''
SELECT p.name, t.jobtitle, e.salary
FROM employeedetails e
JOIN jobtitles t ON e.jobtitleid = t.jobtitleid
JOIN personaldetails p ON e.ssn = p.ssn
JOIN departments d ON e.departmentid = d.departmentid
WHERE e.salary < 100000 AND d.departmentname = 'Engineering'
ORDER BY e.salary DESC;
''')

,name,jobtitle,salary
0,Jennifer Bonnie Duffy,Engineer Level 4,99980.47
1,Luis Christopher Holder,Engineer Level 3,99890.86
2,Michelle Mary Davis,Engineer Level 1,99860.47
3,Lisa Crystal Austin,Engineer Level 3,99838.28
4,Catherine Deanna Hunter,Engineer Level 4,99820.89
...,...,...,...
391,Brian Shawn Parrish,Engineer Level 1,81480.18
392,Billy Daniel Jenkins,Engineer Level 1,81478.14
393,Kaitlyn Amanda Allen,Engineer Level 1,81471.96
394,Jeremiah Jared Marshall,Engineer Level 1,81424.51


**Q3.** Human Resources employees who work at the division located in **Austin**: name, job title and division. Find the division through its city; do not type the division's name.

In [25]:
# Q3
q('''
SELECT p.name, t.jobtitle, d.divtype
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
JOIN divisions d ON e.divisionid = d.divisionid
JOIN divisionlocations l ON e.divisionid = l.divisionid
JOIN jobtitles t ON e.jobtitleid = t.jobtitleid
JOIN departments dpt ON e.departmentid = dpt.departmentid
WHERE l.divcity = 'Austin' AND dpt.departmentname = 'Human Resources';
''')

,name,jobtitle,divtype
0,Daniel Nicholas Parker,Human Resources Specialist 1,South Division
1,Nicole Carly Hicks,Human Resources Specialist 1,South Division
2,Laura Amanda Gonzalez,Human Resources Specialist 4,South Division
3,Morgan Katelyn Jacobson,Human Resources Specialist 1,South Division
4,Raymond Andrew Wade,Human Resources Manager Level 1,South Division
...,...,...,...
124,Kenneth Jeff Stout,Human Resources Specialist 3,South Division
125,Charles Robert Jackson,Human Resources Specialist 3,South Division
126,Jessica Pamela Knight,Human Resources Specialist 3,South Division
127,James Kyle Evans,Human Resources Manager Level 3,South Division


**Q4.** Human Resources only: for each division, the number of HR employees and their average salary (2 decimals), highest average first.

In [26]:
# Q4
q('''
SELECT div.divtype, COUNT(*) AS employees, ROUND(AVG(e.salary),2) AS avg_salary
FROM employeedetails e
JOIN departments dpt ON e.departmentid = dpt.departmentid
JOIN divisions div ON e.divisionid = div.divisionid
WHERE dpt.departmentname = 'Human Resources' 
GROUP BY div.divtype
ORDER BY avg_salary DESC;
''')

,divtype,employees,avg_salary
0,East Division,129,53445.78
1,North Division,129,53045.74
2,Central Division,129,52765.92
3,West Division,129,52742.41
4,South Division,129,52523.17


**Q5.** Engineering employees whose salary is at or above the average Engineering salary: name, job title, salary. The average must be computed inside the query.

In [27]:
# Q5
q('''
SELECT p.name, t.jobtitle, e.salary
FROM employeedetails e
JOIN jobtitles t ON e.jobtitleid = t.jobtitleid
JOIN personaldetails p ON e.ssn = p.ssn
JOIN departments d ON e.departmentid = d.departmentid
WHERE d.departmentname = 'Engineering' 
    AND e.salary >= (
    SELECT AVG(e2.salary)
    FROM employeedetails e2
    JOIN departments d2 ON e2.departmentid = d2.departmentid
    WHERE d2.departmentname = 'Engineering'
    )
ORDER BY e.salary ASC;
''')

,name,jobtitle,salary
0,Colleen Katherine Gentry,Engineer Level 1,108306.47
1,Patricia Jennifer Miller,Engineer Level 1,108323.75
2,Eric Michael Velez,Engineer Level 1,108376.49
3,Douglas Mark Anderson,Engineer Level 1,108401.88
4,Michael Jeffrey Petersen,Engineer Level 4,108514.87
...,...,...,...
456,Samantha Nicole Snyder,Lead Engineering Manager,208455.58
457,Rodney Kyle Anderson,Lead Engineering Manager,223912.18
458,Jillian Danielle Bray,Lead Engineering Manager,231262.00
459,Timothy Matthew Ortega,Lead Engineering Manager,262471.31


**Q6.** Employees aged 50 or older versus under 50 (age as of today): for each group, the number of employees and the average salary. Leave out anyone with no birthdate.

In [28]:
# Q6
q('''

SELECT CASE WHEN EXTRACT(YEAR FROM AGE(CURRENT_DATE, p.birthdate)) >= 50
            THEN '50 or older' ELSE 'Under 50' END AS age_group,
       COUNT(*) AS employees,
       ROUND(AVG(e.salary), 2) AS avg_salary  
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
WHERE p.birthdate IS NOT NULL
  AND p.birthdate <> '1900-01-01'   
GROUP BY age_group
ORDER BY age_group;
''')

,age_group,employees,avg_salary
0,50 or older,3254,46928.29
1,Under 50,11015,43891.17


**Q7.** For each division: its city and state, its headcount and its average salary, largest headcount first.

In [29]:
# Q7
q('''
SELECT d.divtype, l.divcity, l.divstate, 
    COUNT (*) AS employees, 
    ROUND(AVG(e.salary), 2) AS avg_salary
FROM employeedetails e
JOIN divisions d ON e.divisionid = d.divisionid
JOIN divisionlocations l ON e.divisionid = l.divisionid
GROUP BY d.divtype, l.divcity, l.divstate
ORDER BY employees DESC;
''')

,divtype,divcity,divstate,employees,avg_salary
0,South Division,Austin,TX,2868,45542.14
1,West Division,Sacramento,CA,2863,44329.77
2,North Division,Springfield,IL,2863,44340.42
3,East Division,Albany,NY,2863,44233.35
4,Central Division,Columbus,OH,2863,44427.61


**Q8.** Departments whose average salary is higher than the company-wide average salary, with that department average.

In [30]:
# Q8
q('''
SELECT d.departmentname, ROUND(AVG(e.salary), 2) AS avg_salary
FROM employeedetails e
JOIN departments d ON e.departmentid = d.departmentid
GROUP BY d.departmentname
HAVING AVG(e.salary) > (SELECT AVG(salary) FROM employeedetails)
ORDER BY avg_salary ASC;
''')

,departmentname,avg_salary
0,Human Resources,52904.60
1,Engineering,108286.63
2,Management,227950.60
3,Executive,756238.76


**Q9.** The highest-paid employee in each department: department, name, job title and salary.

In [31]:
# Q9
q('''
SELECT departmentname, name, jobtitle, salary FROM (
    SELECT d.departmentname, p.name, t.jobtitle, e.salary,
           ROW_NUMBER() OVER (PARTITION BY d.departmentname
                              ORDER BY e.salary DESC) AS rnk
    FROM employeedetails e
    JOIN personaldetails p ON e.ssn = p.ssn
    JOIN departments d ON e.departmentid = d.departmentid
    JOIN jobtitles t ON e.jobtitleid = t.jobtitleid
) ranked
WHERE rnk = 1
ORDER BY departmentname;
''')

,departmentname,name,jobtitle,salary
0,Administrative,Christopher Justin Stanley,Secretary Level 8,55967.39
1,Engineering,Lindsey Breanna Villa,Lead Engineering Manager,291487.46
2,Executive,Mark Alexander Hill,Chief Executive Officer,1111605.37
3,Human Resources,Michelle Angela Conley,Lead HR Manager,192030.36
4,Management,Darryl Taylor Heath,Division Manager,249261.08
5,Operations,Veronica Rebecca Mclaughlin,Lead Operations Manager,293491.71


**Q10.** Data quality: every email address that belongs to more than one person, with how many people share it. Then answer in the markdown cell below.

In [32]:
# Q10
q('''
SELECT p.email, COUNT(*) AS people_sharing
FROM personaldetails p
WHERE p.email IS NOT NULL
GROUP BY p.email
HAVING COUNT(*) > 1
ORDER BY people_sharing DESC, p.email;
''')

,email,people_sharing
0,anthony_thomas_200@KLB.com,2
1,cheryl_peterson_139@KLB.com,2
2,christopher_young_182@KLB.com,2
3,james_smith_138@KLB.com,2
4,jessica_anderson_140@KLB.com,2
5,joseph_davis_144@KLB.com,2
6,kevin_jones_171@KLB.com,2
7,luis_davis_161@KLB.com,2
8,melissa_moore_147@KLB.com,2
9,michael_morgan_159@KLB.com,2


**Q10, part 2 (2 to 3 sentences):** Look at the rows Q10 returned. Is a shared email a data error, or could it be legitimate? KLB wants to use email as each employee's login. What would you check or change in the data or the schema before that is safe?

> The shared emails are a data error. When two people have the same name it is possible for their emails to be the same as well with the cleaning process I used. I would chnage this by adding an extra step that re-generates the random numbers for each pair of emails. 

# Part C: Dashboard

Your dashboard is `app.py`, a separate file. Build it from the specification in the instructions PDF, run it with `streamlit run app.py`, and put your screenshots and answers in the PDF. Nothing for Part C goes in this notebook.

# Part D: Industry scenarios

These are the kind of incidents data engineers handle at large companies. There is no single right answer. Answer each one in the same cell, under the question, in 3 to 5 sentences. Refer to **your own** loader, SQL or `app.py` wherever you can: a specific answer (the line, the query, the setting) beats a general one.

**D1. The job said SUCCESS.** Your loader runs every night. For months it has rejected a handful of rows each night. One Monday it rejects about 4,100 of 14,000 rows, the job still reports SUCCESS, and the dashboard's headcount drops by almost a third. Nobody notices for two days. The cause: the HR system started sending salaries as `"$85,000.00"` instead of `85000`.

1. Look at how **your** loader converts `Salary`. What would it do with `"$85,000.00"`, and where would those rows end up? If your loader would behave differently from the story above, say how.
2. Why did nothing fail?
3. What would you add to the loader so that next time this stops the run or alerts someone?


> With this current loader, the same issue would occur as there is no process that strips the salaries. Nothing would have failed becasue the salary would have been marked as invlaid in the  `df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")` check, and added to the `rejects_employees.csv` file. In the future, there should be a step while cleaning data where the salary is extracted as a float if it is entered as a string (most likely using regrex). Even with this implemented, it would be a good practice to add alerts to the load process.

**D2. Two numbers for the same thing.** Finance's report and your dashboard both read the same `EmployeeDB`, but they show different average salaries for Engineering. Your manager asks which one is wrong.

1. Give three legitimate reasons the two numbers could differ without either query having a bug. At least one reason must come from how your own `app.py` works.
2. How would you settle which number the company should use?

> 1. `@st.cache_data(ttl=600)`; Data can be up to 10 minutes old, causing a difference in average salaries if data was recently uploaded. 

> 2. There could be filters applied to the dashboard.

> 3. My process selects "Engineering" from the department, but finance could be filtering "Engineering" from job titles. 

> - I would remove all filters from the dashboard (2), make sure my and finance's queries either locate engineers from the department name or the job title, not both (3), and run the query again on the same snapshot (1). 

**D3. From 14,000 rows to 50 million.** After KLB acquires another company, the nightly file grows from about 14,000 rows to 50 million.

1. Time one run of your loader on your own data (for example, wrap one of the runs in A4 with `time.perf_counter()`) and report the time.
2. What in your loader would break or slow down first at 50 million rows? Point to the specific part of your code and explain why it gets slow.
3. What would you change?


> 1. 16.8932 seconds.  

> 2. The first place that the loader would breakdown/slow is while doing the last step, `employeedetails`. This is because each run searches through all of the rows while checking for duplicates by scanning every `employeeid`. 

> 3. I would add a `UNIQUE` constraint for the employeeid. 